# HWP + Eval 정답 기반 청크 메타데이터 재구축

두 가지 소스를 결합하여 청크의 예산/기간 메타데이터를 정확하게 업데이트한다.

1. eval CSV `ground_truth_answer` → 파일명별 예산/기간 정답 추출 (가장 신뢰도 높음)
2. HWP 원본 텍스트 → eval에 없는 파일 보완
3. 두 소스 중 eval 정답 우선 적용


In [1]:
import json, re, subprocess, shutil, glob
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

HWP_DIR    = Path('/mnt/gukrul/dataset/original_data_list/files_advanced')
CHUNK_PATH = Path('/mnt/gukrul/dataset/chunks/kh_fixed_with_budget.json')
EVAL_DIR   = Path('/mnt/gukrul/dataset/eval')

print(f'HWP 파일 수: {len(list(HWP_DIR.glob("*.hwp")))}')
print(f'eval 파일 수: {len(list(EVAL_DIR.glob("eval_batch_*.csv")))}')


HWP 파일 수: 665
eval 파일 수: 38


In [2]:
# eval CSV 전체 로드
files = sorted(EVAL_DIR.glob('eval_batch_*.csv'))
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(f'eval 총 행: {len(df)}')
print(df['type'].value_counts().to_string())


eval 총 행: 1100
type
A    328
B    311
D    162
E    156
C    143


In [3]:
BUDGET_RE   = re.compile(r'([\d,]+(?:\.\d+)?)\s*원')
DURATION_RE = re.compile(r'(\d+)\s*개월')

def extract_from_answer(answer: str):
    budget = duration = None
    bm = BUDGET_RE.search(str(answer))
    dm = DURATION_RE.search(str(answer))
    if bm:
        try:
            val = int(bm.group(1).replace(',', ''))
            if 1_000_000 <= val <= 1_000_000_000_000:
                budget = val
        except: pass
    if dm:
        val = int(dm.group(1))
        if 1 <= val <= 120:
            duration = val
    return budget, duration

# 파일명 → (budget, duration) 매핑 테이블 생성
file_meta = {}  # {원본파일명: {'budget': ..., 'duration': ...}}

for _, row in df.iterrows():
    ans  = str(row.get('ground_truth_answer', ''))
    docs_raw = row.get('ground_truth_docs', '[]')
    try:
        docs = json.loads(docs_raw) if pd.notna(docs_raw) else []
    except:
        continue

    budget, duration = extract_from_answer(ans)

    for doc in docs:
        if doc not in file_meta:
            file_meta[doc] = {'budget': None, 'duration': None}
        if budget and not file_meta[doc]['budget']:
            file_meta[doc]['budget'] = budget
        if duration and not file_meta[doc]['duration']:
            file_meta[doc]['duration'] = duration

budget_count   = sum(1 for v in file_meta.values() if v['budget'])
duration_count = sum(1 for v in file_meta.values() if v['duration'])
print(f'eval 파일 수: {len(file_meta)}')
print(f'예산 확보: {budget_count}개')
print(f'기간 확보: {duration_count}개')

# 가스공사 ERP 확인
key = '한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp'
print(f'\n가스공사 ERP: {file_meta.get(key)}')


eval 파일 수: 91
예산 확보: 67개
기간 확보: 7개

가스공사 ERP: {'budget': 14107009000, 'duration': None}


In [4]:
# HWP에서 보완 추출
def extract_hwp_text(hwp_path):
    try:
        r = subprocess.run(['hwp5txt', str(hwp_path)],
                           capture_output=True, text=True, timeout=30)
        return r.stdout
    except:
        return ''

HWP_BUDGET_PATTERNS = [
    (r'([\d,]+)\s*원\s*\(부가가치세\s*포함\)', 'won'),
    (r'([\d,]+)\s*원\s*\(VAT\s*포함\)', 'won'),
    (r'([\d,]+)\s*원\s*\(부가세\s*포함\)', 'won'),
    (r'소요\s*예산[^\d]*([\d,]+)\s*원', 'won'),
    (r'추정\s*가격[^\d]*([\d,]+)\s*원', 'won'),
    (r'추정금액[^\d]*([\d,]+)\s*원', 'won'),
    (r'총\s*사업비[^\d]*([\d,]+)\s*원', 'won'),
    (r'(?:사업비|예산|계약\s*금액)[^\d]*([\d,]+)\s*원', 'won'),
    (r'([\d,]+(?:\.\d+)?)\s*억\s*원', 'eok'),
    (r'([\d,]+)\s*백만\s*원', 'baengman'),
]

HWP_DURATION_PATTERNS = [
    r'계약(?:일|체결일|일로부터)[^\d]*(\d+)\s*개월',
    r'사업기간[^\d]*(\d+)\s*개월',
    r'용역기간[^\d]*(\d+)\s*개월',
    r'수행기간[^\d]*(\d+)\s*개월',
    r'(\d+)\s*개월\s*(?:간|이내|이하|내)',
]

def parse_hwp_budget(text):
    for pat, unit in HWP_BUDGET_PATTERNS:
        for m in re.finditer(pat, text):
            raw = m.group(1).replace(',', '')
            try:
                val = float(raw)
                if unit == 'won':       result = int(val)
                elif unit == 'eok':    result = int(val * 100_000_000)
                elif unit == 'baengman': result = int(val * 1_000_000)
                if 1_000_000 <= result <= 1_000_000_000_000:
                    return result
            except: continue
    return None

def parse_hwp_duration(text):
    for pat in HWP_DURATION_PATTERNS:
        m = re.search(pat, text)
        if m:
            val = int(m.group(1))
            if 1 <= val <= 120:
                return val
    return None


In [5]:
# HWP로 보완 (eval에 없는 파일)
hwp_files = {f.name: f for f in HWP_DIR.glob('*.hwp')}
hwp_supplemented = 0

for hwp_name, hwp_path in tqdm(hwp_files.items()):
    meta = file_meta.get(hwp_name, {'budget': None, 'duration': None})
    if meta['budget'] and meta['duration']:
        continue  # eval에서 이미 확보

    text = extract_hwp_text(hwp_path)
    if not text.strip():
        continue

    if not meta['budget']:
        b = parse_hwp_budget(text)
        if b:
            meta['budget'] = b
            hwp_supplemented += 1

    if not meta['duration']:
        d = parse_hwp_duration(text)
        if d:
            meta['duration'] = d

    file_meta[hwp_name] = meta

budget_count   = sum(1 for v in file_meta.values() if v['budget'])
duration_count = sum(1 for v in file_meta.values() if v['duration'])
print(f'HWP 보완: {hwp_supplemented}건')
print(f'최종 예산 확보: {budget_count}개')
print(f'최종 기간 확보: {duration_count}개')


100%|██████████| 665/665 [00:00<00:00, 3667.67it/s]

HWP 보완: 0건
최종 예산 확보: 67개
최종 기간 확보: 7개


In [6]:
# 청크 로드
with open(CHUNK_PATH, encoding='utf-8') as f:
    chunks = json.load(f)
print(f'총 청크 수: {len(chunks)}')

# original_name 기준 인덱스 매핑
name_to_indices = defaultdict(list)
for i, c in enumerate(chunks):
    orig = c.get('metadata', {}).get('original_name', '')
    if orig:
        name_to_indices[orig].append(i)
print(f'고유 원본 파일 수: {len(name_to_indices)}')


총 청크 수: 16248
고유 원본 파일 수: 690


In [8]:
updated_budget = updated_duration = 0

for orig_name, indices in name_to_indices.items():
    meta_src = file_meta.get(orig_name, {})
    new_budget   = meta_src.get('budget')
    new_duration = meta_src.get('duration')

    for idx in indices:
        meta = chunks[idx].get('metadata', {})

        if new_budget:
            if meta.get('budget') != new_budget:
                updated_budget += 1
            meta['budget']      = new_budget
            meta['budget_만원'] = round(new_budget / 10_000, 1)

        if new_duration:
            if meta.get('duration_months') != new_duration:
                updated_duration += 1
            meta['duration_months'] = new_duration

    # text 필드 업데이트
            b = meta.get('budget')
            d = meta.get('duration_months')
            budget_str = f"{b/1e8:.1f}억원({int(b/10000):,}만원)" if b and not (isinstance(b, float) and b != b) else '미확인'
            dur_str    = f"{int(d)}개월" if d and not (isinstance(d, float) and d != d) else '미확인'
            proj       = meta.get('project_name', '')
            chunks[idx]['text'] = f'[{proj}] 사업예산: {budget_str} 사업기간: {dur_str}'
print(f'예산 업데이트: {updated_budget}건')
print(f'기간 업데이트: {updated_duration}건')


예산 업데이트: 443건
기간 업데이트: 224건


In [9]:
# 백업 후 저장
backup = CHUNK_PATH.with_suffix('.json.bak')
shutil.copy(CHUNK_PATH, backup)
print(f'백업: {backup}')

with open(CHUNK_PATH, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)
print(f'저장 완료: {CHUNK_PATH}')


백업: /mnt/gukrul/dataset/chunks/kh_fixed_with_budget.json.bak
저장 완료: /mnt/gukrul/dataset/chunks/kh_fixed_with_budget.json


In [10]:
budget_ok = sum(1 for c in chunks if c.get('metadata', {}).get('budget'))
dur_ok    = sum(1 for c in chunks if c.get('metadata', {}).get('duration_months'))
print(f'예산 있음: {budget_ok}/{len(chunks)} ({budget_ok/len(chunks)*100:.1f}%)')
print(f'기간 있음: {dur_ok}/{len(chunks)} ({dur_ok/len(chunks)*100:.1f}%)')

# 가스공사 ERP 확인
hits = [c for c in chunks if '가스공사' in c.get('metadata', {}).get('agency', '') and 'ERP' in c.get('text', '')]
if hits:
    h = hits[0]
    print()
    print('text:', h.get('text', ''))
    print('budget:', h['metadata'].get('budget'))
    print('duration_months:', h['metadata'].get('duration_months'))


예산 있음: 16213/16248 (99.8%)
기간 있음: 282/16248 (1.7%)


## 다음 단계: ChromaDB 재인덱싱

```bash
pkill -f uvicorn
nohup python3 /mnt/gukrul/rebuild_chroma_v2.py > /mnt/gukrul/rebuild_chroma.log 2>&1 &
tail -f /mnt/gukrul/rebuild_chroma.log
```

완료 후:
```bash
bash /mnt/gukrul/start.sh
```
